# Phishing Email Analyser - round 4: campaign-aware split

Same recipe as the shipped round 1 (public corpora + NLLB Arabic translations + 1,199 Qwen 2.5 7B pairs), retrained on
a split that keeps every phishing campaign - the same text under different subjects - on ONE side. The data arrives
already normalised and split by the .NET trainer, so this notebook only trains.

**Settings:** Accelerator **GPU T4 x2** · Internet **On** · attach the dataset `phishing-round4-bundle`. **Run All**,
then download `results.zip` from the Output tab.

In [ ]:
# 1. Unpack the bundle (Kaggle keeps an uploaded zip either unpacked or as a .zip - handle both)
import glob, os, shutil, zipfile
PROJECT = '/kaggle/working/project'
if not os.path.exists(f'{PROJECT}/ml/train_transformer.py'):
    unpacked = glob.glob('/kaggle/input/**/ml/train_transformer.py', recursive=True)
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if unpacked:
        shutil.copytree(os.path.dirname(os.path.dirname(unpacked[0])), PROJECT, dirs_exist_ok=True)
    elif zips:
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(PROJECT)
    else:
        raise FileNotFoundError('Attach the phishing-round4-bundle dataset (right panel > Add Input)')
os.chdir(PROJECT)
os.environ.update(ML_THERMAL_GUARD='off', PYTHONUNBUFFERED='1')
!pip install -q transformers==4.46.3 onnx==1.17.0 onnxruntime==1.20.1 accelerate==1.1.1
!nvidia-smi --query-gpu=name,memory.total --format=csv
!wc -l data/processed/*.jsonl

In [ ]:
# 2. Fine-tune (3 epochs, same settings as round 1), export int8 ONNX + parity fixtures (~45 min)
!python ml/train_transformer.py --base distilbert/distilbert-base-multilingual-cased --epochs 3 --batch 32

In [ ]:
# 3. Package for download (Output tab -> results.zip)
!cd /kaggle/working/project && zip -qr /kaggle/working/results.zip models/transformer models/transformer-metrics.json
!ls -la /kaggle/working/results.zip
!head -60 models/transformer-metrics.json